# Basic RAG Chatbot

This notebook implements the same RAG pipeline as `app.py`:

1. Load FAQ documents
2. Split into chunks
3. Convert chunks into embeddings
4. Store in ChromaDB (vector database)
5. Retrieve relevant chunks for a question
6. Generate an answer using **only** retrieved context

**Setup:** Create a `.env` file with `OPENROUTER_API_KEY=your-key` ([OpenRouter](https://openrouter.ai/)) and install dependencies from `requirements.txt`.

## 1. Environment and imports

In [1]:
import os
import uuid
from typing import List

import chromadb
from dotenv import load_dotenv
from openai import OpenAI

In [2]:
load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

if not OPENROUTER_API_KEY:
    raise ValueError(
        "OPENROUTER_API_KEY is missing. Get a key at https://openrouter.ai/ "
        "and add it to your .env file."
    )

client = OpenAI(
    base_url=OPENROUTER_BASE_URL,
    api_key=OPENROUTER_API_KEY,
)

## 2. Configuration

In [6]:
EMBEDDING_MODEL = "text-embedding-3-small"
GENERATION_MODEL = "gpt-4o-mini"

CHROMA_DB_PATH = "chroma_db"
COLLECTION_NAME = "faq_rag_collection"

FAQ_FILE_PATH = "../data/Basic_FQA.txt"


def openrouter_model(model: str) -> str:
    """Map short model names to OpenRouter provider/model slugs."""
    if "/" in model:
        return model
    return f"openai/{model}"

## 3. Initialize ChromaDB

In [4]:
chroma_client = chromadb.PersistentClient(path=CHROMA_DB_PATH)

collection = chroma_client.get_or_create_collection(name=COLLECTION_NAME)
print(f"Collection '{COLLECTION_NAME}' ready. Current chunk count: {collection.count()}")

Collection 'faq_rag_collection' ready. Current chunk count: 0


## 4. Load FAQ documents

In [7]:
def load_faq_file(file_path: str) -> str:
    """Loads the FAQ document from a text file."""
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"FAQ file not found: {file_path}")

    with open(file_path, "r", encoding="utf-8") as file:
        return file.read()


faq_text = load_faq_file(FAQ_FILE_PATH)
print(f"Loaded {len(faq_text)} characters from {FAQ_FILE_PATH}")
print("\n--- Preview (first 500 chars) ---\n")
print(faq_text[:500])

Loaded 9767 characters from ../data/Basic_FQA.txt

--- Preview (first 500 chars) ---

Q: What is a RAG chatbot?
A: A RAG chatbot retrieves relevant information from documents before generating an answer.

Q: Why do we use embeddings?
A: Embeddings convert text into vectors so we can search for semantically similar content.

Q: What is a vector database?
A: A vector database stores embeddings and allows similarity search.

Q: What does chunking mean?
A: Chunking means splitting long documents into smaller parts that are easier to search and retrieve.

Q: Why should the chatbot ans


## 5. Split document into chunks

In [8]:
def split_text_into_chunks(
    text: str,
    chunk_size: int = 500,
    overlap: int = 100,
) -> List[str]:
    """Splits long text into overlapping chunks."""
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


chunks = split_text_into_chunks(faq_text)
print(f"Created {len(chunks)} chunks")
print("\n--- First chunk ---\n")
print(chunks[0])

Created 25 chunks

--- First chunk ---

Q: What is a RAG chatbot?
A: A RAG chatbot retrieves relevant information from documents before generating an answer.

Q: Why do we use embeddings?
A: Embeddings convert text into vectors so we can search for semantically similar content.

Q: What is a vector database?
A: A vector database stores embeddings and allows similarity search.

Q: What does chunking mean?
A: Chunking means splitting long documents into smaller parts that are easier to search and retrieve.

Q: Why should the chatbot ans


## 6. Create embeddings

In [9]:
def get_embedding(text: str) -> List[float]:
    """Creates an embedding vector via OpenRouter."""
    response = client.embeddings.create(
        model=openrouter_model(EMBEDDING_MODEL),
        input=text,
    )
    return response.data[0].embedding


# Quick sanity check on one chunk
sample_embedding = get_embedding(chunks[0][:200])
print(f"Embedding dimension: {len(sample_embedding)}")

Embedding dimension: 1536


## 7. Ingest chunks into the vector database

Run this once. If documents are already ingested, the cell skips re-ingestion.

In [10]:
def ingest_faq_documents() -> str:
    """Loads FAQ file, chunks it, embeds each chunk, and stores in ChromaDB."""
    existing_count = collection.count()

    if existing_count > 0:
        return f"Documents already ingested. Current chunks in DB: {existing_count}"

    faq_text = load_faq_file(FAQ_FILE_PATH)
    chunks = split_text_into_chunks(faq_text)

    ids = []
    embeddings = []
    documents = []
    metadatas = []

    for index, chunk in enumerate(chunks):
        ids.append(str(uuid.uuid4()))
        documents.append(chunk)
        embeddings.append(get_embedding(chunk))
        metadatas.append({"source": FAQ_FILE_PATH, "chunk_index": index})

    collection.add(
        ids=ids,
        embeddings=embeddings,
        documents=documents,
        metadatas=metadatas,
    )

    return f"Ingestion completed. Stored {len(chunks)} chunks in ChromaDB."


print(ingest_faq_documents())

Ingestion completed. Stored 25 chunks in ChromaDB.


## 8. Retrieve relevant chunks

In [11]:
def retrieve_relevant_chunks(question: str, top_k: int = 3) -> List[str]:
    """Embeds the user question and retrieves the most relevant chunks."""
    question_embedding = get_embedding(question)

    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=top_k,
    )

    return results["documents"][0]

## 9. Generate answer (grounded in context only)

In [14]:
def generate_answer(question: str, context_chunks: List[str]) -> str:
    """Generates an answer via OpenRouter, grounded only in retrieved context."""
    context = "\n\n".join(context_chunks)

    response = client.chat.completions.create(
        model=openrouter_model(GENERATION_MODEL),
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a helpful RAG chatbot. "
                    "Answer using ONLY the provided context. "
                    "If the answer is not in the context, say exactly: "
                    "I do not know based on the provided documents."
                ),
            },
            {
                "role": "user",
                "content": f"Context:\n{context}\n\nQuestion:\n{question}",
            },
        ],
        temperature=0.2,
    )

    return response.choices[0].message.content

In [ ]:
def generate_answer(question: str, context_chunks: List[str]) -> str:
    """Generates an answer via OpenRouter, grounded only in retrieved context."""
    context = "\n\n".join(context_chunks)

    response = client.chat.completions.create(
        model=openrouter_model(GENERATION_MODEL),
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a helpful RAG chatbot. Answer the question  this: "
                ),
            },
            {
                "role": "user",
                "content": f"Context:\n{context}\n\nQuestion:\n{question}",
            },
        ],
        temperature=0.2,
    )

    return response.choices[0].message.content

## 10. Ask a question

Change `question` below and re-run the cell.

In [15]:
question = "What is a RAG chatbot?"
question = "What is the capital of France?"

if collection.count() == 0:
    print("Run the ingestion cell first (section 7).")
else:
    retrieved_chunks = retrieve_relevant_chunks(question)
    answer = generate_answer(question, retrieved_chunks)

    print("Question:", question)
    print("\n--- Answer ---\n")
    print(answer)
    print("\n--- Retrieved context ---\n")
    for i, chunk in enumerate(retrieved_chunks, start=1):
        print(f"Chunk {i}:\n{chunk}\n")

Question: What is the capital of France?

--- Answer ---

I do not know based on the provided documents.

--- Retrieved context ---

Chunk 1:
chatbot can use PDFs if their text is extracted and added to the knowledge base.

Q: Can a RAG chatbot use websites?
A: Yes, website content can be collected, cleaned, chunked, and added to the knowledge base.

Q: What is a chatbot response?
A: A chatbot response is the answer generated for the user’s question.

Q: Why should responses be clear and short?
A: Clear and short responses are easier for users to understand.

Q: What is the main goal of a basic RAG chatbot?
A: The main goal is to ans

Chunk 2:
ors that represent meaning.

Q: What is a vector?
A: A vector is a list of numbers that represents the meaning of a piece of text.

Q: Why do similar texts have similar embeddings?
A: Similar texts have similar embeddings because the model places related meanings close together in vector space.

Q: What is similarity search?
A: Similarity search 

### Optional: reset the vector database

Run the cell below if you need to re-ingest (e.g. after changing the FAQ file).

In [ ]:
# Uncomment to delete the collection and re-ingest from scratch
# chroma_client.delete_collection(COLLECTION_NAME)
# collection = chroma_client.get_or_create_collection(name=COLLECTION_NAME)
# print(ingest_faq_documents())

### OpenRouter API test

Run after sections 1–2 to verify your `OPENROUTER_API_KEY`.

In [ ]:
# Quick OpenRouter connectivity test (run after sections 1–2)
try:
    chat_response = client.chat.completions.create(
        model=openrouter_model(GENERATION_MODEL),
        messages=[{"role": "user", "content": "Say: OpenRouter API key is working."}],
        max_tokens=30,
    )
    print("Chat test passed:", chat_response.choices[0].message.content)

    embedding_response = client.embeddings.create(
        model=openrouter_model(EMBEDDING_MODEL),
        input="This is a test sentence for embeddings.",
    )
    vector = embedding_response.data[0].embedding
    print("Embedding test passed. Vector length:", len(vector))

except Exception as e:
    print("OpenRouter API test failed.")
    print("Error:", e)